# SymBro — RFdiffusion on Colab (no local GPU required)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-ORG/symbro/blob/main/notebooks/symbro_rfdiffusion_colab.ipynb)

> **TODO before publishing:** swap `YOUR-ORG/symbro` above for the real GitHub URL
> once the repository is public, and fill in the citation block at the bottom.

Use this notebook when you don't have a local GPU and no HPC/SLURM cluster is
reachable. It runs the **exact same RFdiffusion job** you already built and validated
in SymBro — the contigs, hotspots, and design count are read verbatim from a
`job_manifest.json` that SymBro's `colab.prepare_colab_bundle()` generated for you.
Nothing about the run is decided here; this notebook only needs RFdiffusion itself — it
does not need SymBro installed in Colab at all. RFdiffusion installs from
[sokrypton/RFdiffusion](https://github.com/sokrypton/RFdiffusion), the fork behind the
well-known ColabDesign RFdiffusion notebook, since it's kept in sync with Colab's own
environment (current torch/CUDA build) rather than the general-purpose install steps in
RosettaCommons' own README.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or any GPU option
offered to you), then run the cells in order.

**Workflow:** (0) confirm a GPU is attached → (1) install RFdiffusion once per session
→ (2) upload the `.zip` bundle `colab.prepare_colab_bundle()` produced → (3) run
inference, with a live progress bar → (4) preview the designs in 3D → (5) download
`results.zip` and hand it back to `colab.import_colab_results()` on your own machine.

---

## 0. Check a GPU is attached

RFdiffusion hard-requires a real CUDA GPU — it doesn't just run slowly on CPU, it fails
outright. Catching a missing GPU here, before spending several minutes on installs, is
cheaper than finding out during step 3.

In [ ]:
#@title Check GPU is available { display-mode: "form" }
import subprocess

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print("GPU detected:", out.stdout.strip())
except Exception:
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type, pick a GPU (e.g. T4), "
        "then re-run this notebook from the top."
    )

## 1. Install RFdiffusion (once per Colab session, ~3 min)

Model weights and noise schedules download in the background (via `aria2c`, 16-way
parallel — much faster and more resumable than a plain `wget`) while RFdiffusion and its
Python dependencies install in the foreground. This closely follows the setup cell from
[ColabDesign](https://github.com/sokrypton/ColabDesign)'s own RFdiffusion notebook — the
most widely-used, actively-maintained Colab entry point for RFdiffusion — trimmed down to
just RFdiffusion itself (this notebook doesn't need ColabDesign's own contig-building
helpers, symmetry auto-detection, or AlphaFold weights: SymBro already built and
validated this exact job locally, so nothing here re-derives or second-guesses it).

In [ ]:
#@title Convenience packages (progress bar + 3D preview) { display-mode: "form" }
!pip install -q tqdm py3Dmol

In [ ]:
#@title Install RFdiffusion { display-mode: "form" }
import os
import time

REQUIRED_WEIGHTS = ["Base_ckpt.pt"]
OPTIONAL_WEIGHTS = ["Complex_base_ckpt.pt"]  # only needed if job_manifest sets hotspot_res
ALL_WEIGHTS = REQUIRED_WEIGHTS + OPTIONAL_WEIGHTS

if not os.path.isdir("params"):
    os.system("apt-get install -y -q aria2 > /dev/null")
    os.makedirs("params", exist_ok=True)
    # Weights + noise schedules, fetched in the background (16-way parallel via aria2c)
    # while the pip installs below run in the foreground.
    os.system(
        "(\
        aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
        aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
        aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
        touch params/done.txt) &"
    )

if not os.path.isdir("RFdiffusion"):
    print("Installing RFdiffusion...")
    # sokrypton's fork: run_inference.py lives at the repo root (not scripts/), and its
    # dependency pins below are kept in sync with Colab's current torch/CUDA build --
    # the same fork the ColabDesign notebook itself installs.
    os.system("git clone -q https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    # Order matters: RFdiffusion's own setup.py depends on a package literally named
    # "se3-transformer", which doesn't exist on PyPI -- it only resolves once this
    # editable install has registered a local package under that name.
    os.system("cd RFdiffusion/env/SE3Transformer && pip install -q .")
    # --no-dependencies avoids pulling in nvidia-cuda-* packages that fight Colab's own
    # CUDA install.
    os.system("pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx")

if not os.path.isdir("RFdiffusion/models"):
    print("Waiting for model weights + schedules to finish downloading...")
    os.makedirs("RFdiffusion/models", exist_ok=True)
    # aria2c leaves a "<file>.aria2" control file while a download is in progress and
    # removes it on success -- poll for that, bounded, rather than trusting a silent
    # infinite wait if a URL is ever wrong.
    for _ in range(90):  # ~7.5 min ceiling
        if not any(os.path.isfile(f"{m}.aria2") for m in ALL_WEIGHTS):
            break
        time.sleep(5)

    present = [m for m in ALL_WEIGHTS if os.path.isfile(m)]
    missing_required = [m for m in REQUIRED_WEIGHTS if m not in present]
    missing_optional = [m for m in OPTIONAL_WEIGHTS if m not in present]
    if missing_optional:
        print(f"Note: {missing_optional} did not download -- only a problem if "
              f"job_manifest.json sets hotspot_res.")
    if present:
        os.system(f"mv {' '.join(present)} RFdiffusion/models")
    if os.path.isfile("schedules.zip"):
        os.system("unzip -q schedules.zip && rm schedules.zip")
    if missing_required:
        raise FileNotFoundError(
            f"{missing_required} never finished downloading -- these are required for "
            f"every job. Re-run this cell; if it keeps failing, check "
            f"https://github.com/RosettaCommons/RFdiffusion for a moved URL."
        )

os.environ["DGLBACKEND"] = "pytorch"
print("RFdiffusion ready.")

## 2. Upload the SymBro bundle

Upload the `.zip` produced by `colab.prepare_colab_bundle(job)` on your own machine.

In [ ]:
import json
import os
import shutil
import zipfile

from google.colab import files

uploaded = files.upload()  # pick the *_colab_bundle.zip file
bundle_zip = next(iter(uploaded))

if os.path.exists("bundle"):
    shutil.rmtree("bundle")
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall("bundle")

manifest_path = "bundle/job_manifest.json"
if not os.path.exists(manifest_path):
    raise FileNotFoundError(
        f"{bundle_zip!r} doesn't contain job_manifest.json — make sure you uploaded "
        f"the *_colab_bundle.zip that colab.prepare_colab_bundle(job) produced, not "
        f"some other file."
    )

with open(manifest_path) as f:
    manifest = json.load(f)

print("Bundle contents:", os.listdir("bundle"))
print("\ncontigs:     ", manifest["contigs"])
print("hotspot_res: ", manifest["hotspot_res"])
print("num_designs: ", manifest["num_designs"])

## 3. Run RFdiffusion

Reads `job_manifest.json`'s `overrides` list — the exact same Hydra key=value tokens
SymBro's local/Singularity backends would have built — and runs inference with them,
unmodified. Using a Python argv list (not a shell string) sidesteps any bash-escaping
issues with `contigmap.contigs=[...]`'s brackets and embedded spaces.

Runs as a background process with a live progress bar (counting `design_<N>.pdb` files
as RFdiffusion writes them — the same technique SymBro's own `poll_status()` uses
locally), rather than blocking silently until the whole job finishes. Interrupting this
cell (`Runtime -> Interrupt execution`) terminates RFdiffusion cleanly.

In [ ]:
#@title Run inference { display-mode: "form" }
import glob
import os
import re
import subprocess
import time

from tqdm.auto import tqdm

os.makedirs("bundle/output", exist_ok=True)

log_path = "bundle/rfdiffusion.log"
# sokrypton/RFdiffusion keeps run_inference.py at the repo root, not under scripts/.
argv = ["python", "../RFdiffusion/run_inference.py"] + manifest["overrides"]

pattern = re.compile(r"design_\d+\.pdb$")
num_designs = manifest["num_designs"]

os.chdir("bundle")
try:
    with open("rfdiffusion.log", "w") as log_file:
        process = subprocess.Popen(argv, stdout=log_file, stderr=subprocess.STDOUT)

    try:
        with tqdm(total=num_designs, desc="RFdiffusion designs") as pbar:
            seen = 0
            while True:
                returncode = process.poll()
                written = sorted(p for p in glob.glob("output/design_*.pdb") if pattern.search(p))
                if len(written) > seen:
                    pbar.update(len(written) - seen)
                    seen = len(written)
                if returncode is not None:
                    break
                time.sleep(2)
    except KeyboardInterrupt:
        process.terminate()
        print("Cancelled — partial designs (if any) are still in bundle/output/.")
        raise
finally:
    os.chdir("..")

if returncode != 0:
    raise RuntimeError(
        f"RFdiffusion exited with code {returncode} — see {log_path} for the full log "
        f"(e.g. `!tail -50 {log_path}`)."
    )

design_paths = sorted(glob.glob("bundle/output/design_*.pdb"))
print(f"\nDone: {len(design_paths)}/{num_designs} designs written.")

## 4. Preview designs

Quick sanity check before downloading — pick a design from the dropdown to render it
inline.

In [ ]:
#@title Preview a design { display-mode: "form" }
import ipywidgets as widgets
import py3Dmol
from IPython.display import display as ipy_display

def show_structure(path):
    view = py3Dmol.view(width=600, height=400)
    with open(path) as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"cartoon": {"colorscheme": "chainHetatm"}})
    view.zoomTo()
    return view

design_dropdown = widgets.Dropdown(options=design_paths, description="Design:")
preview_area = widgets.Output()

def _on_change(change):
    preview_area.clear_output()
    with preview_area:
        show_structure(change["new"]).show()

design_dropdown.observe(_on_change, names="value")
ipy_display(design_dropdown, preview_area)

if design_paths:
    with preview_area:
        show_structure(design_paths[0]).show()

## 5. Download results

Zips `bundle/output/` (the `design_<N>.pdb`/`.trb` files RFdiffusion just wrote) and
downloads it. Hand `results.zip` to `colab.import_colab_results(job, "results.zip")`
back in SymBro — it will drop the designs at the same `output_prefix` a
local/Singularity run would have used, so the rest of your pipeline (ProteinMPNN, etc.)
doesn't need to know these came from Colab.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("results", "zip", "bundle/output")
print(f"Zipped {len(design_paths)}/{num_designs} design(s) -> results.zip")
files.download("results.zip")

---

## Back in SymBro

```python
result = colab.import_colab_results(job, "results.zip")
```

`result` is shaped exactly like `rfdiffusion.poll_status()`'s return value, so the rest
of your pipeline (ProteinMPNN, folding) doesn't need an "this came from Colab" branch.

## Citation

If this pipeline contributed to a publication, please cite:

- **SymBro** — citation TBD, see `CITATION.cff` in the repository once published.
- **RFdiffusion** — Watson, J.L., Juergens, D., Bennett, N.R. *et al.* De novo design of
  protein structure and function with RFdiffusion. *Nature* 620, 1089–1100 (2023).

## License

RFdiffusion itself is BSD-licensed (RosettaCommons); commercial use is permitted.
SymBro's own license: TBD, see the repository's `LICENSE` file once finalized.